In [ ]:
import requests
import time

# ============================================================
# CONFIGURATION SECTION
# ============================================================

# Base endpoint for agent performance data API
# This API returns daily telephony performance metrics per agent,
# including answered calls, talk time, hold time, wrap time,
# and total handle time for workforce analytics and reporting.
url = "https://api.example.com/v1/agent-performance"

# Authentication endpoint used to retrieve Bearer token
auth_url = "https://api.example.com/auth/token"

# Credentials used to generate access token (should be stored securely in production)
client_id = "<client_id>"
client_secret = "<client_secret>"

# Date range for historical / backfill extraction
start_date = "2026-02-01"
end_date = "2026-02-07"

# Incremental watermark used to fetch only new or updated records
# In production, this is typically stored in a control table (e.g., Delta table)
last_updated = "2026-02-01T00:00:00Z"

# Token validity assumption (API states token expires every 60 minutes)
TOKEN_LIFETIME = 3600

# Rate Limits:
# - Maximum 1,000 requests per hour
# - If exceeded, API returns HTTP 429 (Too Many Requests)
# Handling strategy:
# - Monitor response.status_code == 429
# - Pause (sleep) and retry request
# - Ensure ingestion does not exceed 1,000 calls/hour


# ============================================================
# AUTHENTICATION LOGIC
# ============================================================

def get_token():
    """
    Requests a new Bearer token from authentication service.
    This token is required in every API request header.
    """

    response = requests.post(
        auth_url,
        data={
            "client_id": client_id,
            "client_secret": client_secret
        }
    )

    if response.status_code != 200:
        raise Exception(f"Failed to retrieve token: {response.text}")

    token_data = response.json()

    return token_data["access_token"], time.time()


def is_token_expired(token_time):
    """
    Checks if token has exceeded its valid lifetime (60 minutes).
    """
    return (time.time() - token_time) > TOKEN_LIFETIME


# ============================================================
# INITIAL TOKEN SETUP
# ============================================================

token, token_time = get_token()


# ============================================================
# INGESTION STORAGE
# ============================================================

all_records = []
cursor = None


# ============================================================
# MAIN INGESTION LOOP
# ============================================================

while True:

    if is_token_expired(token_time):
        token, token_time = get_token()

    headers = {
        "Authorization": f"Bearer {token}"
    }

    params = {
        "start_date": start_date,
        "end_date": end_date,
        "last_updated": last_updated
    }

    if cursor:
        params["cursor"] = cursor

    response = requests.get(url, headers=headers, params=params)

    # ========================================================
    # RATE LIMIT HANDLING (HTTP 429)
    # ========================================================
    # If API returns 429:
    # - It means request limit (1,000/hour) has been exceeded
    # - Recommended action: pause execution and retry
    # - In production: implement exponential backoff + retry counter
    #
    # NOTE: This code currently does NOT implement retry logic,
    # but this is where it should be handled.
    if response.status_code == 429:
        raise Exception("Rate limit exceeded (HTTP 429). Retry logic should be implemented here.")

    if response.status_code == 401:
        token, token_time = get_token()
        continue

    if response.status_code != 200:
        raise Exception(f"API failed: {response.status_code} - {response.text}")

    payload = response.json()

    records = payload.get("data", [])
    all_records.extend(records)

    cursor = payload.get("next_cursor")

    if not cursor:
        break


# ============================================================
# FINAL OUTPUT
# ============================================================

print(f"Total records fetched: {len(all_records)}")